# Building Knowledge Stores

Create a knowledge store and add videos and images to build a persistent, queryable collection plus derived understanding. This notebook covers creating stores with different ingestion configurations and adding assets.

In [ ]:
import os
import time

from twelvelabs import (
    TwelveLabs,
    IngestionConfig,
    EnrichmentConfig_Description,
    EnrichmentConfig_JsonSchema,
    EnrichmentConfigJsonSchemaJsonSchema,
)

# Configuration
API_KEY = os.environ.get("TWELVELABS_API_KEY", "YOUR_API_KEY")

client = TwelveLabs(api_key=API_KEY)

## When You Need This

You have assets uploaded and want to organize them into a collection that Jockey can reason over. A knowledge store contains your videos and images plus derived understanding: spatiotemporal context, a typed ontology, and embeddings that enable semantic retrieval and corpus-level reasoning.

## Helper: Wait for Item Indexing

In [ ]:
def wait_for_item_ready(
    store_id: str,
    item_id: str,
    interval: int = 10,
    timeout: int = 600,
):
    """Poll a knowledge store item until it reaches 'ready' or 'failed' status.

    Args:
        store_id: The knowledge store identifier.
        item_id: The item identifier to monitor.
        interval: Seconds between polling attempts.
        timeout: Maximum seconds to wait before raising an error.

    Returns:
        The KnowledgeStoreItem object once it reaches 'ready' status.

    Raises:
        Exception: If the item fails indexing or the timeout is exceeded.
    """
    elapsed = 0
    while elapsed < timeout:
        item = client.knowledge_store_items.retrieve(
            knowledge_store_id=store_id, item_id=item_id
        )

        if item.status == "ready":
            print(f"Item {item_id} is ready.")
            return item
        elif item.status == "failed":
            raise Exception(f"Item {item_id} indexing failed.")

        print(f"Item status: {item.status} (elapsed: {elapsed}s)")
        time.sleep(interval)
        elapsed += interval

    raise Exception(f"Timeout after {timeout}s waiting for item {item_id}.")

## Create a Basic Knowledge Store

The simplest knowledge store requires only a name. Jockey uses default extraction settings.

In [ ]:
store = client.knowledge_stores.create(name="My Video and Image Collection")
STORE_ID = store.id
print(f"Knowledge Store ID: {STORE_ID}")

## Create with Ingestion Config: Natural Language

Shape what Jockey extracts by providing a natural-language description. Jockey interprets your description to guide extraction.

In [ ]:
marketing_store = client.knowledge_stores.create(
    name="Marketing Analysis",
    ingestion_config=IngestionConfig(
        enrichment_config=EnrichmentConfig_Description(
            description=(
                "Focus on brand mentions, product appearances, "
                "audience reactions, and visual tone"
            )
        )
    ),
)
print(f"Marketing Store ID: {marketing_store.id}")

## Create with Ingestion Config: JSON Schema

For precise, structured extraction, provide a JSON Schema to define the exact fields you need.

In [ ]:
# Every property must include a "description" — the platform uses it to guide extraction quality.
security_store = client.knowledge_stores.create(
    name="Security Monitoring",
    ingestion_config=IngestionConfig(
        enrichment_config=EnrichmentConfig_JsonSchema(
            json_schema=EnrichmentConfigJsonSchemaJsonSchema(
                type="object",
                properties={
                    "people_count": {"type": "integer", "description": "Number of people visible in the frame"},
                    "location": {"type": "string", "description": "Name or type of the location"},
                    "suspicious_activity": {"type": "boolean", "description": "Whether suspicious activity is detected"},
                    "scene_description": {"type": "string", "description": "Brief description of what is happening"},
                },
                required=["people_count", "scene_description"],
            )
        )
    ),
)
print(f"Security Store ID: {security_store.id}")

## Add a Video or Image to the Store

Once you have a knowledge store and a `ready` asset, add the asset to the store. Indexing happens asynchronously. By default, the platform treats the asset as a video; set `asset_type` to `image` when adding an image.

In [ ]:
ASSET_ID = "your_asset_id"  # Replace with an actual asset ID

item = client.knowledge_store_items.create(
    knowledge_store_id=STORE_ID,
    asset_id=ASSET_ID,
    # asset_type="image",  # Uncomment to add an image (the default is video)
)
item_id = item.id
print(f"Item ID: {item_id}")

# Wait for indexing to complete
ready_item = wait_for_item_ready(STORE_ID, item_id)
print("Indexed and ready.")

## Add Multiple Videos and Images

Add a batch of assets and wait for all to finish indexing.

In [ ]:
ASSET_IDS = ["asset_1", "asset_2", "asset_3"]  # Replace with actual asset IDs
item_ids: list[str] = []

# Add all assets (asset_type defaults to "video"; pass asset_type="image" per call to add an image)
for asset_id in ASSET_IDS:
    item = client.knowledge_store_items.create(
        knowledge_store_id=STORE_ID,
        asset_id=asset_id,
    )
    item_ids.append(item.id)
    print(f"Added asset {asset_id} -> item {item.id}")

# Wait for all items to be ready
for item_id in item_ids:
    while True:
        status = client.knowledge_store_items.retrieve(
            knowledge_store_id=STORE_ID, item_id=item_id
        ).status
        if status == "ready":
            print(f"Item {item_id} is ready.")
            break
        elif status == "failed":
            print(f"Item {item_id} failed.")
            break
        time.sleep(10)

print(f"All {len(item_ids)} assets processed.")

## Common Pitfalls

- **Asset must be ready first.** Adding a `processing` asset will fail. Wait for the asset to reach `ready` status.
- **`asset_type` must match the asset.** It defaults to `video`; set it to `image` when adding an image, or the request fails.
- **Indexing takes time.** Expect roughly 1-10 minutes per video depending on length; images typically index faster.

## Next Steps

- [Ingestion Config](ingestion_config.ipynb) — learn more about configuring extraction
- [Querying](querying.ipynb) — ask questions about your indexed videos and images
- [Authentication](authentication.ipynb) — API key setup and security

**API Reference:**
- [POST /knowledge-stores](https://twelvelabs-preview-7f6af7ac-b5df-4358-a7dc-8573e931a808.docs.buildwithfern.com/api-reference/knowledge-stores/create-knowledge-store)
- [POST /knowledge-stores/{id}/items](https://twelvelabs-preview-7f6af7ac-b5df-4358-a7dc-8573e931a808.docs.buildwithfern.com/api-reference/knowledge-store-items/create-knowledge-store-item)